# Round 1 Combo Strategy Analysis

This notebook documents the simple observations that led to `ROUND1/pepper_osmium_combo.py`:

- `INTARIAN_PEPPER_ROOT` trends upward almost linearly within each day, so an early long position is extremely valuable.
- `ASH_COATED_OSMIUM` is much closer to a stationary market-making product, and quoting one tick inside only pays when the spread is still comfortably wide.


In [1]:
import pandas as pd
from pathlib import Path

price_paths = sorted(Path('../data/ROUND1').glob('prices_round_1_day_*.csv'))
prices = pd.concat([pd.read_csv(path, sep=';') for path in price_paths], ignore_index=True)
for col in prices.columns:
    if col != 'product':
        prices[col] = pd.to_numeric(prices[col], errors='coerce')
prices = prices.dropna(subset=['bid_price_1', 'ask_price_1']).copy()
prices['mid'] = (prices['bid_price_1'] + prices['ask_price_1']) / 2
prices['spread'] = prices['ask_price_1'] - prices['bid_price_1']
prices.head()


,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss,mid,spread
0,-1,0,INTARIAN_PEPPER_ROOT,10991.0,15.0,NaN,NaN,NaN,NaN,11006.0,10.0,11009.0,15.0,NaN,NaN,10998.5,0.0,10998.5,15.0
2,-1,100,ASH_COATED_OSMIUM,9984.0,11.0,NaN,NaN,NaN,NaN,10000.0,11.0,10003.0,22.0,NaN,NaN,9992.0,0.0,9992.0,16.0
3,-1,100,INTARIAN_PEPPER_ROOT,10994.0,9.0,10991.0,21.0,NaN,NaN,11006.0,9.0,11009.0,21.0,NaN,NaN,11000.0,0.0,11000.0,12.0
4,-1,200,ASH_COATED_OSMIUM,9985.0,15.0,9982.0,20.0,NaN,NaN,10001.0,15.0,10003.0,20.0,NaN,NaN,9993.0,0.0,9993.0,16.0
5,-1,200,INTARIAN_PEPPER_ROOT,10994.0,10.0,NaN,NaN,NaN,NaN,11009.0,20.0,NaN,NaN,NaN,NaN,11001.5,0.0,11001.5,15.0


In [2]:
pepper = prices[prices['product'] == 'INTARIAN_PEPPER_ROOT'].copy()
pepper_summary = pepper.groupby('day').agg(
    start_mid=('mid', 'first'),
    end_mid=('mid', 'last'),
    rows=('mid', 'size')
)
pepper_summary['total_move'] = pepper_summary['end_mid'] - pepper_summary['start_mid']
pepper_summary['slope_per_tick'] = pepper_summary['total_move'] / (pepper_summary['rows'] - 1)
pepper_summary


,start_mid,end_mid,rows,total_move,slope_per_tick
day,,,,,
-2,9998.5,11001.5,9216,1003.0,0.108844
-1,10998.5,11998.0,9219,999.5,0.108429
0,11998.5,13000.0,9253,1001.5,0.108247


In [3]:
osmium = prices[prices['product'] == 'ASH_COATED_OSMIUM'].copy()
osmium.groupby('day')['spread'].agg(['mean', 'median', 'min', 'max']).round(2)


,mean,median,min,max
day,,,,
-2,16.15,16.0,5.0,21.0
-1,16.19,16.0,5.0,21.0
0,16.18,16.0,5.0,22.0


In [4]:
results = pd.DataFrame([
    {'strategy': 'greedy3_reupload', 'day': -2, 'pnl': 92159},
    {'strategy': 'greedy3_reupload', 'day': -1, 'pnl': 92281},
    {'strategy': 'greedy3_reupload', 'day': 0, 'pnl': 92458},
    {'strategy': 'pepper_osmium_combo', 'day': -2, 'pnl': 93996},
    {'strategy': 'pepper_osmium_combo', 'day': -1, 'pnl': 94838},
    {'strategy': 'pepper_osmium_combo', 'day': 0, 'pnl': 93437},
])
results.pivot(index='day', columns='strategy', values='pnl').assign(
    improvement=lambda df: df['pepper_osmium_combo'] - df['greedy3_reupload']
)


strategy,greedy3_reupload,pepper_osmium_combo,improvement
day,,,
-2,92159,93996,1837
-1,92281,94838,2557
0,92458,93437,979
